# Урок 23. Запросы, фильтры и сортировка

9 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-24.ipynb)

---

Выборка данных по условию. Составные условия. Сортировка результата. Знакомство с SQL-запросом SELECT на простых примерах.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-23", name=ФИО, klass=Класс)

## Разбираемся

### Разговор с базой

Таблицы мы разложили, ключи расставили. Осталось главное: доставать
из базы то, что нужно. Для этого есть язык запросов — **SQL**.

Он устроен непривычно. В Python вы пишете, **как** получить ответ:
заведи счётчик, пройди по списку, сравни, добавь. В SQL пишут,
**что** нужно получить, а как — забота базы.

```sql
SELECT фамилия FROM читатели WHERE класс = '9А';
```

Всё. Перебор, сравнение, сборка результата — не ваша работа.

### Из чего состоит запрос

| Часть | Что задаёт | Обязательна |
|---|---|---|
| `SELECT` | какие поля показать | да |
| `FROM` | из какой таблицы | да |
| `WHERE` | условие отбора | нет |
| `ORDER BY` | порядок сортировки | нет |
| `LIMIT` | сколько строк вернуть | нет |

Порядок частей менять нельзя: сначала `SELECT`, потом `FROM`,
потом `WHERE`, и только потом `ORDER BY`.

```sql
SELECT название, цена       -- что
FROM товары                 -- откуда
WHERE цена > 100            -- какие строки
ORDER BY цена DESC          -- в каком порядке
LIMIT 3;                    -- сколько
```

`*` вместо списка полей означает «все поля»: `SELECT * FROM товары`.
В учебных примерах так пишут постоянно, в настоящих программах —
почти никогда: лишние поля тянутся по сети зря.

### Условия

В `WHERE` работают знакомые сравнения и новые слова:

| Условие | Что отбирает |
|---|---|
| `цена > 100` | дороже ста |
| `категория = 'хлеб'` | точное совпадение (кавычки одинарные!) |
| `цена BETWEEN 50 AND 100` | в промежутке, границы включаются |
| `категория IN ('хлеб', 'молоко')` | из списка |
| `название LIKE 'Мо%'` | начинается с «Мо» |
| `скидка IS NULL` | поле не заполнено |

Условия соединяются словами `AND`, `OR`, `NOT` — ровно как
в Python, только по-английски.

> Два подвоха. Равенство в SQL записывается **одним** знаком `=`,
> а не двумя. И текст берётся в **одинарные** кавычки: `'9А'`.

В `LIKE` символ `%` заменяет любое число символов, а `_` — ровно
один. Это те же маски файлов, что были в седьмом классе,
только значки другие.

### Сортировка

```sql
ORDER BY цена          -- по возрастанию (это по умолчанию)
ORDER BY цена DESC     -- по убыванию
ORDER BY категория, цена DESC   -- сначала по категории, внутри — по цене
```

`ASC` — по возрастанию, `DESC` — по убыванию. Первое можно
не писать.

### Подсчёты

| Функция | Что считает |
|---|---|
| `COUNT(*)` | количество строк |
| `SUM(поле)` | сумму |
| `AVG(поле)` | среднее |
| `MIN`, `MAX` | минимум и максимум |

```sql
SELECT COUNT(*) FROM товары WHERE цена > 100;
```

Такой запрос вернёт **одну строку с одним числом** — не список
товаров, а именно счёт. Это ровно наш счётчик из цикла,
только считать будет база.

### Как это выглядит из Python

```python
курсор.execute("SELECT название FROM товары WHERE цена > 100")
строки = курсор.fetchall()      # список кортежей
```

`fetchall()` возвращает список, где каждая строка — **кортеж**,
даже если поле одно: `[('Сыр',), ('Кофе',)]`. Запятая внутри скобок
не опечатка — так записывается кортеж из одного элемента.

Если ответ заведомо одна строка, берут `fetchone()`:

```python
сколько = курсор.execute("SELECT COUNT(*) FROM товары").fetchone()[0]
```

## Смотрим, как это работает

### Пример 1. Готовим базу

In [ ]:
import sqlite3

связь = sqlite3.connect(":memory:")
курсор = связь.cursor()

курсор.execute("""
    CREATE TABLE товары (
        id INTEGER PRIMARY KEY,
        название TEXT,
        категория TEXT,
        цена INTEGER,
        количество INTEGER
    )
""")

курсор.executemany("INSERT INTO товары VALUES (?, ?, ?, ?, ?)", [
    (1, "Хлеб", "выпечка", 45, 30),
    (2, "Молоко", "молочное", 89, 20),
    (3, "Сыр", "молочное", 420, 5),
    (4, "Кофе", "напитки", 350, 12),
    (5, "Чай", "напитки", 180, 18),
    (6, "Батон", "выпечка", 39, 25),
    (7, "Творог", "молочное", 120, 8),
])
связь.commit()

for строка in курсор.execute("SELECT * FROM товары"):
    print(строка)

### Пример 2. Выбор полей

In [ ]:
print("Все названия:")
for строка in курсор.execute("SELECT название FROM товары"):
    print(" ", строка)

print()
print("Заметьте: каждая строка — кортеж. Само значение — строка[0]:")
for строка in курсор.execute("SELECT название FROM товары LIMIT 3"):
    print(" ", строка[0])

### Пример 3. Условия

In [ ]:
запросы = [
    "SELECT название, цена FROM товары WHERE цена > 100",
    "SELECT название FROM товары WHERE категория = 'молочное'",
    "SELECT название, цена FROM товары WHERE цена BETWEEN 50 AND 200",
    "SELECT название FROM товары WHERE категория IN ('напитки', 'выпечка')",
    "SELECT название FROM товары WHERE название LIKE 'Ч%'",
]

for запрос in запросы:
    print(запрос)
    for строка in курсор.execute(запрос):
        print("   ", строка)
    print()

### Пример 4. Составные условия

In [ ]:
запрос = """
    SELECT название, категория, цена
    FROM товары
    WHERE категория = 'молочное' AND цена < 200
"""

print("Молочное дешевле двухсот:")
for строка in курсор.execute(запрос):
    print("   ", строка)

### Пример 5. Сортировка

In [ ]:
print("Самые дорогие сверху:")
for название, цена in курсор.execute(
        "SELECT название, цена FROM товары ORDER BY цена DESC LIMIT 3"):
    print(f"   {название:<8} {цена}")

print()
print("По категории, внутри — по цене:")
for название, категория, цена in курсор.execute(
        "SELECT название, категория, цена FROM товары ORDER BY категория, цена"):
    print(f"   {категория:<10} {название:<8} {цена}")

### Пример 6. Подсчёты

In [ ]:
print("Всего товаров:", курсор.execute("SELECT COUNT(*) FROM товары").fetchone()[0])
print("Дороже 100:   ", курсор.execute(
    "SELECT COUNT(*) FROM товары WHERE цена > 100").fetchone()[0])
print("Средняя цена: ", курсор.execute("SELECT AVG(цена) FROM товары").fetchone()[0])
print("Самая дорогая:", курсор.execute("SELECT MAX(цена) FROM товары").fetchone()[0])

### Пример 7. Запрос против цикла

Одна и та же задача: названия молочных товаров дешевле двухсот.

In [ ]:
через_sql = курсор.execute("""
    SELECT название FROM товары
    WHERE категория = 'молочное' AND цена < 200
    ORDER BY название
""").fetchall()

все = курсор.execute("SELECT название, категория, цена FROM товары").fetchall()
через_цикл = []
for название, категория, цена in все:
    if категория == "молочное" and цена < 200:
        через_цикл.append((название,))
через_цикл.sort()

print("SQL: ", через_sql)
print("Цикл:", через_цикл)
print("Совпало:", через_sql == через_цикл)

Результат один и тот же. Разница в том, что на миллионе строк
база воспользуется индексом и ответит мгновенно, а цикл честно
переберёт всё.

## Пробуем сами

Дальше вы пишете запросы сами. В каждой задаче нужно дописать
функцию: она получает `курсор`, выполняет запрос и возвращает
`fetchall()` — список кортежей.

База та же, что в примерах: таблица `товары` с полями
`id, название, категория, цена, количество`.

### Задача 1. Все названия по порядку

Верните названия всех товаров в порядке возрастания `id`.

In [ ]:
def все_названия(курсор):
    return курсор.execute("...").fetchall()

In [ ]:
si.check("1", все_названия, [
    (курсор, [("Хлеб",), ("Молоко",), ("Сыр",), ("Кофе",),
              ("Чай",), ("Батон",), ("Творог",)]),
])

### Задача 2. Дорогие товары

Верните названия товаров дороже 200, отсортированные по названию.

In [ ]:
def дорогие(курсор):
    return курсор.execute("...").fetchall()

In [ ]:
si.check("2", дорогие, [
    (курсор, [("Кофе",), ("Сыр",)]),
])

### Задача 3. Одна категория

Верните названия и цены товаров категории `молочное`,
отсортированные по цене по возрастанию.

In [ ]:
def молочное(курсор):
    return курсор.execute("...").fetchall()

In [ ]:
si.check("3", молочное, [
    (курсор, [("Молоко", 89), ("Творог", 120), ("Сыр", 420)]),
])

### Задача 4. Два условия

Верните названия товаров, которые стоят дешевле 200 **и** которых
на складе больше 15 штук. Отсортируйте по названию.

In [ ]:
def дешёвые_и_в_наличии(курсор):
    return курсор.execute("...").fetchall()

In [ ]:
si.check("4", дешёвые_и_в_наличии, [
    (курсор, [("Батон",), ("Молоко",), ("Хлеб",), ("Чай",)]),
])

### Задача 5. Сколько подходит

Верните количество товаров дороже 100. Ответ — одно число,
а не список: воспользуйтесь `COUNT(*)` и `fetchone()[0]`.

In [ ]:
def сколько_дороже_ста(курсор):
    return ...

In [ ]:
si.check("5", сколько_дороже_ста, [
    (курсор, 4),
])

### Задача 6. Слово для условия

Какое слово начинает часть запроса, где задаётся условие отбора
строк? Запишите заглавными буквами.

In [ ]:
#@title 🧩 Задача 6. Слово запроса { display-mode: "form" }
#@markdown Впиши слово
слово = "" #@param {type:"string"}

si.ответ("6", слово, "43f94244d00ce481",
         hint="По-английски «где».")

## Домашнее задание

### Домашнее задание 1. Восемь запросов

К той же базе `товары` составьте и проверьте восемь запросов:

1. все товары категории «напитки»;
2. товары дешевле 100, по убыванию цены;
3. три самых дорогих товара;
4. товары, названия которых начинаются на «К»;
5. средняя цена по всей таблице;
6. количество товаров в категории «выпечка»;
7. товары, которых меньше 10 штук на складе;
8. общая стоимость склада: цена, умноженная на количество, по всем
   товарам (`SELECT SUM(цена * количество) FROM товары`).

Запишите запросы и ответы, покажите учителю.

### Домашнее задание 2. Самый дешёвый

Верните название самого дешёвого товара. Ответ — строка.

Подсказка: отсортируйте по цене и возьмите одну строку.

In [ ]:
def самый_дешёвый(курсор):
    return ...

In [ ]:
si.check("дз2", самый_дешёвый, [
    (курсор, "Батон"),
])

### Домашнее задание 3. Порядок по умолчанию

Если написать `ORDER BY цена` без дополнительных слов,
в каком порядке отсортируется результат?

In [ ]:
#@title 🏠 Домашнее 3. Порядок сортировки { display-mode: "form" }
#@markdown Выбери ответ
порядок = "выбери ответ" #@param ["выбери ответ", "по возрастанию", "по убыванию", "случайный"]

si.ответ("дз3", порядок, "715c407e83e0e848",
         hint="Для убывания пришлось бы дописать DESC.")

---

### Любопытно

SQL придумали в IBM в начале 1970-х и назвали SEQUEL — «структурный
английский язык запросов». Название пришлось менять: оно уже было
зарегистрировано британской авиастроительной фирмой. Осталось SQL,
но произносят его до сих пор двояко — «эс-кью-эль» и «сиквел».

Языку полвека, и за это время сменились поколения языков
программирования, а SQL остался почти тем же. Запрос, написанный
в 1975 году, сегодня выполнится без единой правки — похвастаться
таким может мало кто.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 22](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-22.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 24 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-24.ipynb)